In [1]:
import os, sys

# ── Windows Hadoop fix ──────────────────────────────────────────
os.environ['HADOOP_HOME'] = r'C:\hadoop'
os.environ['PATH'] += r';C:\hadoop\bin'

# ── Python worker fix ───────────────────────────────────────────
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

In [2]:
from pyspark.sql import SparkSession  # = SparkContext + SQL + Streaming

spark = (SparkSession.builder
    .appName("MyApp")
    # .master("spark://10.1.11.5:7077")   # or "local[*]"
    .master("local[*]")   # or "local[*]"
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.1")
    .config("spark.driver.memory", "4g")
    .config("spark.executor.memory", "4g")
    .config("spark.executor.cores", "4")
    .config("spark.executor.instances", "2")
    .getOrCreate()
)

In [3]:
data = [
    ("Abraham", "Apples",  5),
    ("Jacob",   "Bananas", 3),
    ("Jacob",   "Oranges", 2),
    ("Benjamin","Apples",  2),
    ("David",   "Bananas", 2),
    ("David",   "Oranges", 5),
    ("Esau",    "Apples",  3),
    ("Esau",    "Bananas", 5),
    ("Ishmael", "Oranges", 4),
    ("Samael",  "Apples",  4),
    ("Samael",  "Bananas", 1),
    ("Samael",  "Oranges", 1),
]
columns = ["Customer", "Item", "Quantity"]
df = spark.createDataFrame(data, columns)

In [4]:
import os
import subprocess, sys
print("Driver Python:", sys.executable)
result = subprocess.run([sys.executable, "--version"], capture_output=True, text=True)
print("Version:", result.stdout or result.stderr)
print("PYSPARK_PYTHON env:", os.environ.get("PYSPARK_PYTHON", "not set"))


Driver Python: C:\Users\ghuy2\Documents\BACHKHOA\HK_252\BigData\kafka\kafka-lab\Scripts\python.exe
Version: Python 3.10.11

PYSPARK_PYTHON env: C:\Users\ghuy2\Documents\BACHKHOA\HK_252\BigData\kafka\kafka-lab\Scripts\python.exe


In [5]:
df.show()         # Print rows (like pandas df.head())
print(df.count()) # Row count 
df.printSchema()  # Column types (like pandas df.info())

+--------+-------+--------+
|Customer|   Item|Quantity|
+--------+-------+--------+
| Abraham| Apples|       5|
|   Jacob|Bananas|       3|
|   Jacob|Oranges|       2|
|Benjamin| Apples|       2|
|   David|Bananas|       2|
|   David|Oranges|       5|
|    Esau| Apples|       3|
|    Esau|Bananas|       5|
| Ishmael|Oranges|       4|
|  Samael| Apples|       4|
|  Samael|Bananas|       1|
|  Samael|Oranges|       1|
+--------+-------+--------+

12
root
 |-- Customer: string (nullable = true)
 |-- Item: string (nullable = true)
 |-- Quantity: long (nullable = true)



In [6]:
from pyspark.sql.functions import col, sum

# APPROACH 1: filter first → then agg
total_apples = (df
    .filter(col("Item") == "Apples")
    .agg(sum("Quantity").alias("Total_Apples"))
    .collect()[0]["Total_Apples"])
print(f"Total Apples sold: {total_apples}")

# APPROACH 2: groupBy → agg → filter
total_apples_2 = (df
    .groupBy("Item")
    .agg(sum("Quantity").alias("Total_Quantity"))
    .filter(col("Item") == "Apples")
    .collect()[0]["Total_Quantity"])
print(f"Total Apples sold (approach 2): {total_apples_2}")

# APPROACH 3: SQL — filter in WHERE
df.createOrReplaceTempView("purchases")
total_apples_3 = spark.sql("""
    SELECT SUM(Quantity) AS Total_Apples
    FROM purchases
    WHERE Item = 'Apples'
""").collect()[0]["Total_Apples"]
print(f"Total Apples sold (approach 3): {total_apples_3}")

# APPROACH 4: SQL — filter in HAVING
total_apples_4 = spark.sql("""
    SELECT Item, SUM(Quantity) AS Total_Quantity
    FROM purchases
    GROUP BY Item
    HAVING Item = 'Apples'
""").collect()[0]["Total_Quantity"]
print(f"Total Apples sold (approach 4): {total_apples_4}")

Total Apples sold: 14
Total Apples sold (approach 2): 14
Total Apples sold (approach 3): 14
Total Apples sold (approach 4): 14


In [7]:
from pyspark.sql.functions import col, sum

# APPROACH 1: filter first → then agg
total_apples = (df
    .filter(col("Item") == "Apples")
    .agg(sum("Quantity").alias("Total_Apples"))
    .collect()[0]["Total_Apples"])
print(f"Total Apples sold: {total_apples}")

# APPROACH 2: groupBy → agg → filter
total_apples_2 = (df
    .groupBy("Item")
    .agg(sum("Quantity").alias("Total_Quantity"))
    .filter(col("Item") == "Apples")
    .collect()[0]["Total_Quantity"])
print(f"Total Apples sold (approach 2): {total_apples_2}")

# APPROACH 3: SQL — filter in WHERE
df.createOrReplaceTempView("purchases")
total_apples_3 = spark.sql("""
    SELECT SUM(Quantity) AS Total_Apples
    FROM purchases
    WHERE Item = 'Apples'
""").collect()[0]["Total_Apples"]
print(f"Total Apples sold (approach 3): {total_apples_3}")

# APPROACH 4: SQL — filter in HAVING
total_apples_4 = spark.sql("""
    SELECT Item, SUM(Quantity) AS Total_Quantity
    FROM purchases
    GROUP BY Item
    HAVING Item = 'Apples'
""").collect()[0]["Total_Quantity"]
print(f"Total Apples sold (approach 4): {total_apples_4}")

Total Apples sold: 14
Total Apples sold (approach 2): 14
Total Apples sold (approach 3): 14
Total Apples sold (approach 4): 14


In [8]:
price_data = [("Apples", 1.0), ("Bananas", 0.5), ("Oranges", 0.8)]
price_columns = ["Item", "Price"]
price_df = spark.createDataFrame(price_data, price_columns)
price_df.show()

+-------+-----+
|   Item|Price|
+-------+-----+
| Apples|  1.0|
|Bananas|  0.5|
|Oranges|  0.8|
+-------+-----+



In [9]:
from pyspark.sql.functions import expr

# Join on "Item" column (inner join)
joined_df = df.join(price_df, on="Item", how="inner")

# Add computed column: Total_Cost = Quantity * Price
joined_df = joined_df.withColumn("Total_Cost", expr("Quantity * Price"))

# Filter Jacob and sum his costs
jacob_total = (joined_df
    .filter(col("Customer") == "Jacob")
    .agg(sum("Total_Cost").alias("Jacob_Total"))
    .collect()[0]["Jacob_Total"])
print(f"Jacob's total grocery spending: ${jacob_total:.2f}")

Jacob's total grocery spending: $3.10


In [10]:
hc = spark.sparkContext._jsc.hadoopConfiguration()

# S3A library for AWS/MinIO

# MinIO endpoint and credentials
hc.set("fs.s3a.endpoint",   "http://10.1.11.5:9000")
hc.set("fs.s3a.access.key", "test")
hc.set("fs.s3a.secret.key", "hcmuthpcc")

# Path-style access (required for MinIO)
hc.set("fs.s3a.path.style.access", "true")
hc.set("fs.s3a.aws.credentials.provider",
       "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")
hc.set("fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")

# Magic committer — better than default FileOutputCommitter
hc.set("fs.s3a.committer.name", "magic")
hc.set("fs.s3a.committer.magic.partitioned.enabled", "true")
hc.set("fs.s3a.fast.upload",        "true")
hc.set("fs.s3a.fast.upload.buffer", "disk")

# Optional: overwrite and abort pending
hc.set("fs.s3a.committer.staging.conflict-mode",          "replace")
hc.set("fs.s3a.committer.staging.abort.pending.uploads",  "true")

In [11]:
# df.write.mode("overwrite").csv(
#     "s3a://big-data/test/jacob.csv",
#     header=True
# )
df.write.mode("overwrite").csv("./output/jacob.csv", header=True)

In [12]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_json
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, DoubleType, LongType
)

BROKERS = "localhost:9092,localhost:9192,localhost:9292"

ratings_schema = StructType([
    StructField("userId",    IntegerType()),
    StructField("movieId",   IntegerType()),
    StructField("rating",    DoubleType()),
    StructField("timestamp", LongType()),
])

movies_schema = StructType([
    StructField("movieId", IntegerType()),
    StructField("title",   StringType()),
    StructField("genres",  StringType()),
])

tags_schema = StructType([
    StructField("userId",    IntegerType()),
    StructField("movieId",   IntegerType()),
    StructField("tag",       StringType()),
    StructField("timestamp", LongType()),
])

def read_topic(topic, schema):
    raw = (spark.read
        .format("kafka")
        .option("kafka.bootstrap.servers", BROKERS)
        .option("subscribe", topic)
        .option("startingOffsets", "earliest")
        .load())
    return (raw
        .selectExpr("CAST(value AS STRING) AS json_str")
        .select(from_json(col("json_str"), schema).alias("data"))
        .select("data.*"))

df_ratings = read_topic("ratings", ratings_schema)
df_movies  = read_topic("movies",  movies_schema)
df_tags    = read_topic("tags",    tags_schema)

df_ratings.show(3)
df_movies.show(3)
df_tags.show(3)

+------+-------+------+---------+
|userId|movieId|rating|timestamp|
+------+-------+------+---------+
|     1|      1|   4.0|964982703|
|     1|      3|   4.0|964981247|
|     1|      6|   4.0|964982224|
+------+-------+------+---------+
only showing top 3 rows

+-------+--------------------+--------------------+
|movieId|               title|              genres|
+-------+--------------------+--------------------+
|      1|    Toy Story (1995)|Adventure|Animati...|
|      2|      Jumanji (1995)|Adventure|Childre...|
|      3|Grumpier Old Men ...|      Comedy|Romance|
+-------+--------------------+--------------------+
only showing top 3 rows

+------+-------+---------------+----------+
|userId|movieId|            tag| timestamp|
+------+-------+---------------+----------+
|     2|  60756|          funny|1445714994|
|     2|  60756|Highly quotable|1445714996|
|     2|  60756|   will ferrell|1445714992|
+------+-------+---------------+----------+
only showing top 3 rows



In [13]:
from pyspark.sql.functions import avg, count

movie_stats = (df_ratings
    .groupBy("movieId")
    .agg(
        avg("rating").alias("avg_rating"),
        count("rating").alias("rating_count")
    )
    .filter(col("rating_count") > 30)          # only movies with >30 ratings
    .orderBy(col("avg_rating").desc())            # sort highest first
    .limit(5))

# Join with movie titles
top5_movies = movie_stats.join(df_movies, on="movieId", how="left")
top5_movies.select("movieId", "title", "avg_rating", "rating_count").show(truncate=False)

+-------+--------------------------------+-----------------+------------+
|movieId|title                           |avg_rating       |rating_count|
+-------+--------------------------------+-----------------+------------+
|318    |Shawshank Redemption, The (1994)|4.429022082018927|317         |
|1204   |Lawrence of Arabia (1962)       |4.3              |45          |
|858    |Godfather, The (1972)           |4.2890625        |192         |
|2959   |Fight Club (1999)               |4.272935779816514|218         |
|1276   |Cool Hand Luke (1967)           |4.271929824561403|57          |
+-------+--------------------------------+-----------------+------------+



In [14]:
# Join tags with ratings on movieId
tags_with_ratings = df_tags.join(df_ratings, on="movieId", how="inner")

# Group by tag & compute average rating of tagged movies
worst_tags = (tags_with_ratings
    .groupBy("tag")
    .agg(avg("rating").alias("avg_rating"))
    .orderBy(col("avg_rating").asc())   # lowest first
    .limit(5))

worst_tags.show(truncate=False)

+--------+------------------+
|tag     |avg_rating        |
+--------+------------------+
|symbolic|0.5               |
|shark   |1.4166666666666667|
|stage   |1.75              |
|Tokyo   |2.0               |
|SNL     |2.1               |
+--------+------------------+



In [15]:
# Collect the 5 worst tag strings
worst_tag_list = [row["tag"] for row in worst_tags.collect()]
print("Worst tags:", worst_tag_list)

# Filter tags table to only those worst tags
filtered_tags = df_tags.filter(col("tag").isin(worst_tag_list))

# Join with ratings to get each movie's ratings
tagged_movies_ratings = filtered_tags.join(df_ratings, on="movieId", how="inner")

# Per-movie average rating AND how many movies share each tag
movie_tag_stats = (tagged_movies_ratings
    .groupBy("tag", "movieId")
    .agg(
        avg("rating").alias("movie_avg_rating"),
        count("rating").alias("n_ratings")
    )
    .orderBy("tag", col("movie_avg_rating").asc()))

# Join with movie titles for readability
movie_tag_stats = movie_tag_stats.join(
    df_movies.select("movieId", "title"), on="movieId", how="left"
)
movie_tag_stats.show(30, truncate=False)

tag_movie_count = (filtered_tags
    .groupBy("tag")
    .agg(count("movieId").alias("movie_count")))
print("Movies per worst tag:")
tag_movie_count.show(truncate=False)

Worst tags: ['symbolic', 'shark', 'stage', 'Tokyo', 'SNL']
+-------+--------+------------------+---------+------------------------------+
|movieId|tag     |movie_avg_rating  |n_ratings|title                         |
+-------+--------+------------------+---------+------------------------------+
|2296   |SNL     |2.1               |10       |Night at the Roxbury, A (1998)|
|8943   |stage   |1.75              |2        |Being Julia (2004)            |
|26717  |symbolic|0.5               |1        |Begotten (1990)               |
|1389   |shark   |1.4166666666666667|6        |Jaws 3-D (1983)               |
|6407   |Tokyo   |2.0               |1        |Walk, Don't Run (1966)        |
+-------+--------+------------------+---------+------------------------------+

Movies per worst tag:
+--------+-----------+
|tag     |movie_count|
+--------+-----------+
|Tokyo   |1          |
|shark   |1          |
|SNL     |1          |
|symbolic|1          |
|stage   |1          |
+--------+-----------+


In [18]:
from pyspark.sql.functions import avg, count, col
from scipy import stats

# Compare average ratings worst-tagged movies vs the rest

# Get all movieIds that are associated with the worst tags
worst_tag_movie_ids = filtered_tags.select("movieId").distinct()

# Ratings of movies WITH worst tags
ratings_worst = df_ratings.join(worst_tag_movie_ids, on="movieId", how="inner")

# Ratings of movies WITHOUT worst tags
ratings_others = df_ratings.join(worst_tag_movie_ids, on="movieId", how="left_anti")

avg_worst = ratings_worst.agg(avg("rating").alias("avg_rating")).collect()[0]["avg_rating"]
avg_others = ratings_others.agg(avg("rating").alias("avg_rating")).collect()[0]["avg_rating"]

print("=== Average Rating Comparison ===")
print(f"Movies with worst tags : {avg_worst:.4f}")
print(f"All other movies       : {avg_others:.4f}")
print(f"Difference             : {avg_worst - avg_others:.4f}")


# ─────────────────────────────────────────────

# Independent samples t-test

# Collect rating values as Python lists
worst_ratings_list = [row["rating"] for row in ratings_worst.select("rating").collect()]
other_ratings_list = [row["rating"] for row in ratings_others.select("rating").collect()]

t_stat, p_value = stats.ttest_ind(worst_ratings_list, other_ratings_list)

print("\n=== T-Test Result ===")
print(f"T-statistic : {t_stat:.4f}")
print(f"P-value     : {p_value:.6f}")

if p_value < 0.05:
    print("\n Result is statistically significant (p < 0.05).")
    print("   Movies with worst tags DO tend to receive lower ratings.")
else:
    print("\n Result is NOT statistically significant (p >= 0.05).")
    print("   We cannot conclude that worst tags correlate with lower ratings.")

=== Average Rating Comparison ===
Movies with worst tags : 1.7750
All other movies       : 3.5019
Difference             : -1.7269

=== T-Test Result ===
T-statistic : -7.4091
P-value     : 0.000000

 Result is statistically significant (p < 0.05).
   Movies with worst tags DO tend to receive lower ratings.


## Does the "Worst tags" means movies with certain tags Tend to receive lower ratings?

Not necessarily at first glance. While the five worst tags have the lowest average ratings
overall, this does not automatically mean that every movie carrying those tags is poorly
received. To answer this properly, we need to look deeper at the per-movie ratings and how
many movies are actually associated with each tag.

### What the "Worst tags" metric actually tells us

When we rank tags by average rating, we are averaging the ratings of **all movies**
linked to that tag. This means a single extremely poorly rated movie can drag an entire
tag's average down — even if the other movies under that tag are rated perfectly fine.

- Some worst tags are attached to only a **small number of movies**. In these cases,
  one or two low-rated films heavily skew the tag's average, making the tag look bad
  when in reality there is not enough data to support that conclusion.

- For tags with a **larger number of associated movies**, if most of those movies
  consistently show low average ratings, then there is stronger evidence that the tag
  correlates with poorer reception.

- In some cases, movies under the same tag show a **wide spread of ratings**, meaning
  some are rated well and others poorly. This suggests the tag itself has little
  predictive power over how a movie is received.

### Statistical Proof

To move beyond observation, we compared the average ratings of movies associated with
the worst tags against all other movies, and ran an independent samples t-test to verify
whether the difference was statistically meaningful.

| Metric | Value |
|---|---|
| Average rating — worst-tagged movies | 1.7750 |
| Average rating — all other movies | 3.5019 |
| Difference | -1.7269 |
| T-statistic | -7.4091 |
| P-value | < 0.000001 |

The results are striking. Movies associated with the worst tags averaged a rating of
**1.78**, nearly **1.73 points lower** than all other movies at **3.50**. With a
p-value effectively equal to zero and well below the 0.05 significance threshold,
this difference is statistically significant and cannot be attributed to random chance.

### Conclusion

In this case, **yes** — the worst tags result does carry real meaning. Movies associated
with these five tags genuinely tend to receive significantly lower ratings from viewers.
The gap of nearly 1.73 points on a 5-point scale is not a statistical artifact; it is a
strong and consistent signal confirmed by the t-test.

This tells us that these tags are not just unlucky labels dragged down by one or two
outliers. Rather, they appear to be legitimately associated with content that audiences
rate poorly. 